# 6. tēma — tīmekļa datu ieguve: īss atkārtojums

[![Atvērt Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValRCS/RTU_BDAA_Course_2026/blob/main/notebooks/lecture_06_web_scraping/06_01_web_scraping_refresher.ipynb)

Šī darba burtnīca ir **īss web scraping atkārtojums**, nevis pilnīgi jauna tēma.  
Mērķis ir atjaunot praktiskās iemaņas pirms nākamās darba burtnīcas par dzīvokļu sludinājumu datiem.

Pēc darba burtnīcas pabeigšanas jūs atkārtosiet:

- HTTP pieprasījumu ar `requests`;
- HTML dokumenta parsēšanu ar `BeautifulSoup`;
- `find()`, `find_all()` un HTML atribūtus;
- CSS selektorus ar `select()` / `select_one()`;
- atkārtotu elementu pārvēršanu Python vārdnīcās;
- rezultātu pārvēršanu `pandas.DataFrame`;
- CSV faila saglabāšanu.

Mācību mērķim izmantosim vietni **quotes.toscrape.com**, kas ir īpaši paredzēta scraping vingrinājumiem.


## Darbināšana VS Code un Google Colab

### Lokāli ar VS Code
1. Noklonējiet vai lejupielādējiet kursa repozitoriju.
2. Atveriet šo `.ipynb` failu VS Code.
3. Pārliecinieties, ka ir instalēti **Python** un **Jupyter** paplašinājumi.
4. Augšējā labajā stūrī izvēlieties Python kodolu (`kernel`).
5. Palaidiet šūnas secīgi no augšas uz leju.

### Google Colab
Atveriet darba burtnīcu ar **Open in Colab** pogu augstāk un palaidiet šūnas secīgi.

Nākamā sagatavošanas šūna instalēs trūkstošās pakotnes tikai tad, ja tās nav atrodamas pašreizējā Python vidē.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
}

missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Instalējam trūkstošās pakotnes:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Visas nepieciešamās pakotnes jau ir instalētas.")

print("Python:", sys.version.split()[0])
print("Vide:", "Google Colab" if "google.colab" in sys.modules else "lokāls Jupyter/VS Code")


In [ ]:
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

pd.set_option("display.max_colwidth", 100)

BASE_URL = "https://quotes.toscrape.com/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; RTU-BDAA-teaching-example/1.0)"
}


## 1. HTTP pieprasījums

Web scraping parasti sākas ar HTTP pieprasījumu.

`requests.get()` atgriež `Response` objektu. Pirms parsēšanas ir vērts pārbaudīt:

- statusa kodu;
- gala URL;
- `Content-Type`;
- vai pieprasījums nav beidzies ar kļūdu.

`raise_for_status()` izmet izņēmumu 4xx/5xx kļūdu gadījumā.


In [ ]:
response = requests.get(BASE_URL, headers=HEADERS, timeout=15)
response.raise_for_status()

print("Status:", response.status_code)
print("URL:", response.url)
print("Content-Type:", response.headers.get("content-type"))
print("HTML garums:", len(response.text))
print()
print(response.text[:300])


## 2. HTML parsēšana ar BeautifulSoup

Pārvēršam saņemto HTML tekstu objektu struktūrā, kuru var meklēt.

Atkārtojam galvenās metodes:

- `find()` — pirmais atbilstošais elements;
- `find_all()` — visi atbilstošie elementi;
- `.text` vai `.get_text()` — redzamais teksts;
- `.get("href")`, `.get("class")` — HTML atribūti.


In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

print("Lapas title:", soup.title.get_text(strip=True))

first_quote = soup.find("div", class_="quote")
first_quote


In [ ]:
text_element = first_quote.find("span", class_="text")
author_element = first_quote.find("small", class_="author")
tag_elements = first_quote.find_all("a", class_="tag")

print("Citāts:", text_element.get_text(strip=True))
print("Autors:", author_element.get_text(strip=True))
print("Tagi:", [tag.get_text(strip=True) for tag in tag_elements])


## 3. HTML atribūti un saites

HTML elementos dati bieži atrodas ne tikai tekstā, bet arī atribūtos.

Piemēram:

```html
<a href="/author/Albert-Einstein">about</a>
```

Šeit `/author/Albert-Einstein` ir `href` atribūta vērtība.


In [ ]:
about_link = first_quote.find("a")
print("Relatīvā saite:", about_link.get("href"))
print("Pilna saite:", urljoin(BASE_URL, about_link.get("href")))


## 4. CSS selektori

BeautifulSoup atbalsta arī CSS selektorus.

Tie ir īpaši noderīgi, jo to pašu domāšanas veidu vēlāk izmantosim Selenium:

- `.quote` — elements ar klasi `quote`;
- `.quote .text` — `text` klases elements `quote` iekšpusē;
- `a.tag` — `<a>` elements ar klasi `tag`.


In [ ]:
quote_boxes = soup.select("div.quote")
print("Citātu skaits lapā:", len(quote_boxes))

first_text_css = soup.select_one("div.quote span.text")
print(first_text_css.get_text(strip=True))


## 5. Atkārtotu elementu pārvēršana strukturētos datos

Praktiskā scraping uzdevumā parasti negribam saglabāt BeautifulSoup objektus.
Mēs gribam iegūt vienkāršus Python datus — piemēram, vārdnīcas (`dict`).

Viena vārdnīca = viens ieraksts.


In [ ]:
records = []

for quote in soup.select("div.quote"):
    text = quote.select_one("span.text").get_text(strip=True)
    author = quote.select_one("small.author").get_text(strip=True)
    tags = [tag.get_text(strip=True) for tag in quote.select("a.tag")]

    records.append({
        "quote": text,
        "author": author,
        "tags": ", ".join(tags),
    })

print("Iegūti ieraksti:", len(records))
records[:2]


## 6. No Python ierakstiem uz Pandas DataFrame

`DataFrame` ir ērts nākamais solis, jo pēc tam datus var:

- filtrēt;
- kārtot;
- grupēt;
- tīrīt;
- eksportēt uz CSV/Excel.

Detalizētāka datu tīrīšana būs 7. tēmas saturs.


In [ ]:
df = pd.DataFrame(records)
df.head()


In [ ]:
print("Forma:", df.shape)
print()
print("Citāti pa autoriem:")
display(df["author"].value_counts().head(10))


## 7. Neliela atkārtošana: vairākas lapas

Scraping bieži nozīmē **pagination** — datu nolasīšanu no vairākām rezultātu lapām.

Šeit apzināti nolasīsim tikai pirmās 3 lapas un starp pieprasījumiem ieliksim nelielu pauzi.

> Reālās vietnēs vienmēr ievērojiet vietnes noteikumus, `robots.txt` un neveidojiet nevajadzīgi daudz pieprasījumu.


In [ ]:
import time

all_records = []
next_url = BASE_URL

for page_number in range(1, 4):
    print(f"Nolasām lapu {page_number}: {next_url}")

    r = requests.get(next_url, headers=HEADERS, timeout=15)
    r.raise_for_status()
    page_soup = BeautifulSoup(r.text, "html.parser")

    for quote in page_soup.select("div.quote"):
        all_records.append({
            "quote": quote.select_one("span.text").get_text(strip=True),
            "author": quote.select_one("small.author").get_text(strip=True),
            "tags": ", ".join(
                tag.get_text(strip=True) for tag in quote.select("a.tag")
            ),
            "page": page_number,
        })

    next_link = page_soup.select_one("li.next a")
    if next_link is None:
        break

    next_url = urljoin(next_url, next_link.get("href"))
    time.sleep(0.5)

quotes_df = pd.DataFrame(all_records)
print("Kopā iegūti ieraksti:", len(quotes_df))
quotes_df.head()


## 8. Saglabāšana CSV

Izveidosim `outputs` mapi un saglabāsim tajā rezultātu.

VS Code gadījumā fails būs repozitorija darba mapē.  
Colab gadījumā fails būs Colab sesijas failu sistēmā.


In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

output_file = OUTPUT_DIR / "quotes_scraped.csv"
quotes_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Saglabāts:", output_file.resolve())


## 9. Īss praktiskais uzdevums

Pamēģiniet patstāvīgi:

1. nolasīt **5** lapas, nevis 3;
2. pievienot kolonnu `author_url`;
3. atrast, kurš autors iegūtajos datos parādās visbiežāk;
4. atlasīt tikai citātus ar tagu `life`;
5. saglabāt rezultātu jaunā CSV failā.

### Ko vajadzētu atcerēties pirms nākamās darba burtnīcas

Tipiska scraping plūsma:

**URL → HTTP response → HTML → BeautifulSoup → atrast elementus → iegūt vērtības → records → DataFrame → fails**

Nākamajā darba burtnīcā šo pašu pieeju izmantosim reālākam biznesa datu piemēram — dzīvokļu sludinājumiem.
